<a href="https://colab.research.google.com/github/r00trahaf/Arabic-AI-Text-Detection-MSIS-822Project/blob/main/notebooks/MSIS822.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

import pandas as pd
from datasets import load_dataset

In [2]:
DATASET_NAME = "KFUPM-JRCAI/arabic-generated-abstracts"
SELECTED_SPLIT = "from_title_and_content"

BASE_PATH = Path("/content")

RAW_DATA_PATH = BASE_PATH / "data" / "raw"
PROCESSED_DATA_PATH = BASE_PATH / "data" / "processed"

RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

In [3]:
print("Loading dataset from Hugging Face")
print("=" * 70)

dataset = load_dataset(DATASET_NAME)

print(dataset)

Loading dataset from Hugging Face


README.md:   0%|          | 0.00/8.10k [00:00<?, ?B/s]

data/by_polishing-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 8.49MB            

data/by_polishing-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/from_title-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.90MB            

data/from_title-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (2 outputs left out)

Generating by_polishing split:   0%|          | 0/2851 [00:00<?, ? examples/s]

Generating from_title split:   0%|          | 0/2963 [00:00<?, ? examples/s]

Generating from_title_and_content split:   0%|          | 0/2574 [00:00<?, ? examples/s]

DatasetDict({
    by_polishing: Dataset({
        features: ['original_abstract', 'allam_generated_abstract', 'jais_generated_abstract', 'llama_generated_abstract', 'openai_generated_abstract'],
        num_rows: 2851
    })
    from_title: Dataset({
        features: ['original_abstract', 'allam_generated_abstract', 'jais_generated_abstract', 'llama_generated_abstract', 'openai_generated_abstract'],
        num_rows: 2963
    })
    from_title_and_content: Dataset({
        features: ['original_abstract', 'allam_generated_abstract', 'jais_generated_abstract', 'llama_generated_abstract', 'openai_generated_abstract'],
        num_rows: 2574
    })
})


In [4]:
print("Dataset Structure")
print("=" * 70)

print(f"Available subsets: {list(dataset.keys())}")

for split_name in dataset.keys():
    print(f"\nSubset: {split_name}")
    print(f"Number of rows: {len(dataset[split_name])}")
    print(f"Features: {dataset[split_name].features}")

Dataset Structure
Available subsets: ['by_polishing', 'from_title', 'from_title_and_content']

Subset: by_polishing
Number of rows: 2851
Features: {'original_abstract': Value('string'), 'allam_generated_abstract': Value('string'), 'jais_generated_abstract': Value('string'), 'llama_generated_abstract': Value('string'), 'openai_generated_abstract': Value('string')}

Subset: from_title
Number of rows: 2963
Features: {'original_abstract': Value('string'), 'allam_generated_abstract': Value('string'), 'jais_generated_abstract': Value('string'), 'llama_generated_abstract': Value('string'), 'openai_generated_abstract': Value('string')}

Subset: from_title_and_content
Number of rows: 2574
Features: {'original_abstract': Value('string'), 'allam_generated_abstract': Value('string'), 'jais_generated_abstract': Value('string'), 'llama_generated_abstract': Value('string'), 'openai_generated_abstract': Value('string')}


In [5]:
print(f"Selected subset: {SELECTED_SPLIT}")
print("=" * 70)

selected_dataset = dataset[SELECTED_SPLIT]

print(f"Number of raw samples: {len(selected_dataset)}")
print(f"Features: {selected_dataset.features}")

Selected subset: from_title_and_content
Number of raw samples: 2574
Features: {'original_abstract': Value('string'), 'allam_generated_abstract': Value('string'), 'jais_generated_abstract': Value('string'), 'llama_generated_abstract': Value('string'), 'openai_generated_abstract': Value('string')}


In [6]:
df_raw = selected_dataset.to_pandas()

print("\nRaw DataFrame shape:")
print(df_raw.shape)

print("\nColumns:")
print(df_raw.columns.tolist())

print("\nData types:")
print(df_raw.dtypes)


Raw DataFrame shape:
(2574, 5)

Columns:
['original_abstract', 'allam_generated_abstract', 'jais_generated_abstract', 'llama_generated_abstract', 'openai_generated_abstract']

Data types:
original_abstract            object
allam_generated_abstract     object
jais_generated_abstract      object
llama_generated_abstract     object
openai_generated_abstract    object
dtype: object


In [7]:
raw_file = RAW_DATA_PATH / f"{SELECTED_SPLIT}.csv"

df_raw.to_csv(
    raw_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nRaw data saved to:")
print(raw_file)


Raw data saved to:
/content/data/raw/from_title_and_content.csv


In [9]:
print("RAW DATA QUALITY CHECKS")
print("All checks are performed before reshaping.")
print("=" * 70)

RAW DATA QUALITY CHECKS
All checks are performed before reshaping.


In [10]:
print("\n--- Missing Values ---")

missing_values = df_raw.isnull().sum()

print(missing_values)

print(
    f"\nTotal missing values: "
    f"{missing_values.sum()}"
)


--- Missing Values ---
original_abstract            0
allam_generated_abstract     0
jais_generated_abstract      0
llama_generated_abstract     0
openai_generated_abstract    0
dtype: int64

Total missing values: 0


In [11]:
print("\n--- Duplicate Rows ---")

duplicate_rows = df_raw.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows}")



--- Duplicate Rows ---
Duplicate rows: 0


In [12]:
print("\n--- Empty Text Values ---")

text_columns = [
    "original_abstract",
    "allam_generated_abstract",
    "jais_generated_abstract",
    "llama_generated_abstract",
    "openai_generated_abstract"
]

empty_values = {}

for column in text_columns:
    empty_count = (
        df_raw[column]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    empty_values[column] = empty_count

    print(f"{column}: {empty_count}")


--- Empty Text Values ---
original_abstract: 0
allam_generated_abstract: 0
jais_generated_abstract: 0
llama_generated_abstract: 0
openai_generated_abstract: 0


In [13]:
print("\n--- Data Types ---")

print(df_raw.dtypes)

non_string_values = {}

for column in text_columns:
    count = (~df_raw[column].map(lambda x: isinstance(x, str))).sum()
    non_string_values[column] = count

print("\nNon-string values:")
for column, count in non_string_values.items():
    print(f"{column}: {count}")


--- Data Types ---
original_abstract            object
allam_generated_abstract     object
jais_generated_abstract      object
llama_generated_abstract     object
openai_generated_abstract    object
dtype: object

Non-string values:
original_abstract: 0
allam_generated_abstract: 0
jais_generated_abstract: 0
llama_generated_abstract: 0
openai_generated_abstract: 0


In [14]:
print("\n--- Text Length Check ---")

for column in text_columns:

    lengths = df_raw[column].fillna("").astype(str).str.len()

    print(f"\n{column}")
    print(f"Minimum length: {lengths.min()}")
    print(f"Maximum length: {lengths.max()}")
    print(f"Empty texts: {(lengths == 0).sum()}")


--- Text Length Check ---

original_abstract
Minimum length: 416
Maximum length: 1891
Empty texts: 0

allam_generated_abstract
Minimum length: 187
Maximum length: 13235
Empty texts: 0

jais_generated_abstract
Minimum length: 180
Maximum length: 2968
Empty texts: 0

llama_generated_abstract
Minimum length: 257
Maximum length: 1636
Empty texts: 0

openai_generated_abstract
Minimum length: 567
Maximum length: 920
Empty texts: 0


In [15]:
print("\n" + "=" * 70)
print("RAW DATA QUALITY SUMMARY")
print("=" * 70)

print(f"Raw rows: {len(df_raw)}")
print(f"Raw columns: {len(df_raw.columns)}")
print(f"Missing values: {missing_values.sum()}")
print(f"Duplicate rows: {duplicate_rows}")

total_empty_values = sum(empty_values.values())

print(f"Empty text values: {total_empty_values}")


RAW DATA QUALITY SUMMARY
Raw rows: 2574
Raw columns: 5
Missing values: 0
Duplicate rows: 0
Empty text values: 0


In [18]:
print("RESHAPING DATASET")


# Human-written abstracts
human_df = pd.DataFrame({
    "text": df_raw["original_abstract"],
    "label": "Human"
})

# AI-generated abstracts
ai_columns = [
    "allam_generated_abstract",
    "jais_generated_abstract",
    "llama_generated_abstract",
    "openai_generated_abstract"
]

ai_parts = []

for column in ai_columns:

    temp = pd.DataFrame({
        "text": df_raw[column],
        "label": "AI"
    })

    ai_parts.append(temp)


ai_df = pd.concat(
    ai_parts,
    ignore_index=True
)


# Combine Human and AI
df_final = pd.concat(
    [human_df, ai_df],
    ignore_index=True
)

RESHAPING DATASET


In [19]:
print("FINAL CLASSIFICATION DATASET")
print("=" * 70)

print(f"Final shape: {df_final.shape}")

print("\nColumns:")
print(df_final.columns.tolist())

print("\nFirst five rows:")
print(df_final.head())

FINAL CLASSIFICATION DATASET
Final shape: (12870, 2)

Columns:
['text', 'label']

First five rows:
                                                text  label
0  كثيرا ما ارتبطت المصادر التاريخية في الأندلس خ...  Human
1  يعد العامل الثقافي احد ابرز الاسباب التي يعزى ...  Human
2  شكلت تلك الجهود والمساعي الرائدة التي قام بها ...  Human
3  يقوم المقال على اشكالية الضرائب الغير شرعية في...  Human
4  تعددت صور الحياة الثقافية و الفكرية و الأدبية ...  Human


In [20]:
print("LABEL DISTRIBUTION")
print("=" * 70)

label_counts = df_final["label"].value_counts()

label_percentages = (
    df_final["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nCounts:")
print(label_counts)

print("\nPercentages:")
print(label_percentages)

LABEL DISTRIBUTION

Counts:
label
AI       10296
Human     2574
Name: count, dtype: int64

Percentages:
label
AI       80.0
Human    20.0
Name: proportion, dtype: float64


In [21]:
print("FINAL DATASET CHECKS")
print("=" * 70)

print(
    f"Missing text values: "
    f"{df_final['text'].isnull().sum()}"
)

print(
    f"Missing labels: "
    f"{df_final['label'].isnull().sum()}"
)

print(
    f"Non-string text values: "
    f"{(~df_final['text'].map(lambda x: isinstance(x, str))).sum()}"
)

print(
    f"Texts shorter than 10 characters: "
    f"{(df_final['text'].str.len() < 10).sum()}"
)


FINAL DATASET CHECKS
Missing text values: 0
Missing labels: 0
Non-string text values: 0
Texts shorter than 10 characters: 0


In [22]:
print("TEXT LENGTH STATISTICS")
print("=" * 70)

text_lengths = df_final["text"].str.len()

print(text_lengths.describe())

TEXT LENGTH STATISTICS
count    12870.000000
mean       684.971562
std        243.486708
min        180.000000
25%        547.250000
50%        675.000000
75%        780.000000
max      13235.000000
Name: text, dtype: float64


In [23]:
processed_file = (
    PROCESSED_DATA_PATH /
    "from_title_and_content_classification.csv"
)

df_final.to_csv(
    processed_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 70)
print("DATA PREPARATION COMPLETED")
print("=" * 70)

print(f"Raw dataset:")
print(raw_file)

print(f"\nFinal classification dataset:")
print(processed_file)

print(f"\nFinal number of rows: {len(df_final)}")
print(f"Final number of columns: {len(df_final.columns)}")


DATA PREPARATION COMPLETED
Raw dataset:
/content/data/raw/from_title_and_content.csv

Final classification dataset:
/content/data/processed/from_title_and_content_classification.csv

Final number of rows: 12870
Final number of columns: 2
